Problem Statement

The goal of this project is to recommend books to users based on their previous reading interactions. Before building the recommendation system, I first explored the dataset to understand how users were actually behaving on the platform.

One interesting thing I found during EDA was that most users were interacting with only one chapter from a book and then moving to other books instead of reading many chapters from the same book. On average, users were exploring around 6 to 7 different books.

Because of this pattern, I felt that predicting the “next chapter” inside a book would not be the right approach for this dataset. Instead, it made more sense to recommend new books that match a user's reading interests and genre preferences

Assumptions

While working on this project, I made a few basic assumptions based on the available data.

First, I assumed that if a user interacted with a chapter, it means they showed some level of interest in that content. Since the dataset does not contain ratings, likes, or completion percentage, every interaction was treated equally.

Second, I assumed that genres are a strong indicator of user preference. For example, if a user mostly interacts with Crime, Thriller, or Mystery content, there is a good chance they may prefer similar books in future recommendations as well.

I also assumed that books already interacted with by the user should not be recommended again because the user has already discovered those books.

Finally, for completely new users where no interaction history is available, I used popular books as a fallback recommendation strategy

Approach

For this project, I used a content-based recommendation approach.

The main reason for choosing this approach was the nature of the dataset. During EDA, I found that users had very limited interaction history. Most users interacted with only a few books, which makes the interaction data quite sparse. Because of this, collaborative filtering would not work very reliably since many users do not have enough overlapping behavior.

Instead of comparing users with each other, content-based filtering focuses on understanding user taste using the content information available in the books.

To build the recommendation system, I first created a genre profile for every user using the genres from books they interacted with. Similarly, I also created a genre profile for every book.

After creating these profiles, I used cosine similarity to compare a user’s taste profile with all available books. Books with higher similarity scores were considered more relevant for that user.

Before generating the final recommendations, books already interacted with by the user were removed so that the system only suggests new books.

For users with no interaction history, the system recommends the most popular books on the platform as a cold-start fallback approach.

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

loading datasets
chapters.csv contains book and chapter information
interactions.csv contains user reading interactions

In [2]:
chapters = pd.read_csv('chapters.csv')
interactions = pd.read_csv('interactions.csv')

print('chapters shape:', chapters.shape)
print('interactions shape:', interactions.shape)

chapters shape: (50000, 6)
interactions shape: (1000000, 3)


converting published date into datetime format
this is not heavily used later
but keeping date columns in correct format is good practice

In [3]:
chapters['published_date'] = pd.to_datetime( chapters['published_date'], errors='coerce' )

BASIC DATA UNDERSTANDING
============================================================
before building the model,
I want to understand how users are behaving.


this is important because recommendation systems depend heavily
on user behavior patterns.

In [4]:
print('\nchecking first few rows from chapters data')
print(chapters.head())

print('\nchecking first few rows from interactions data')
print(interactions.head())


checking first few rows from chapters data
   chapter_id  chapter_sequence_no  book_id  author_id published_date  \
0     2812946                    1   139726      66847     1990-03-22   
1     4330764                    2   139726      66847     1990-04-09   
2     2664499                    3   139726      66847     1990-04-07   
3     2260666                    4   139726      66847     1990-05-18   
4     6069976                    1   191772      62262     2008-07-30   

                                       tags  
0                            Fantasy|Horror  
1      Fantasy|Young Adult|Literary Fiction  
2                                   Fantasy  
3                  Literary Fiction|Fantasy  
4  Horror|Young Adult|Romance|Graphic Novel  

checking first few rows from interactions data
        user_id  chapter_id  book_id
0  user_2378720     5894067   444295
1  user_2321122     2532511   785684
2  user_2335775     6777764   999595
3  user_7906001     7366896   748410
4  user_

checking missing values
because null values can create problems later

In [5]:
print('\nmissing values in chapters data')
print(chapters.isnull().sum())

print('\nmissing values in interactions data')
print(interactions.isnull().sum())


missing values in chapters data
chapter_id             0
chapter_sequence_no    0
book_id                0
author_id              0
published_date         0
tags                   0
dtype: int64

missing values in interactions data
user_id       0
chapter_id    0
book_id       0
dtype: int64


EDA - UNDERSTANDING USER BEHAVIOR


here I am checking:
- how many books users read
- how many chapters users read
- whether users are reading books sequentially


this is important because the recommendation strategy depends on this.
number of books per user

In [6]:
books_per_user = ( interactions .groupby('user_id')['book_id'] .nunique() )

print('\nbooks per user statistics')
print(books_per_user.describe())


books per user statistics
count    149803.000000
mean          6.672230
std           2.570475
min           1.000000
25%           5.000000
50%           6.000000
75%           8.000000
max          20.000000
Name: book_id, dtype: float64


number of chapters per user

In [7]:
chapters_per_user = ( interactions .groupby('user_id')['chapter_id'] .nunique() )

print('\nchapters per user statistics')
print(chapters_per_user.describe())


chapters per user statistics
count    149803.000000
mean          6.675434
std           2.572317
min           1.000000
25%           5.000000
50%           6.000000
75%           8.000000
max          20.000000
Name: chapter_id, dtype: float64


now checking chapters read inside each book
this is the most important analysis for this assignment


if users are reading many chapters inside same book,
then sequential recommendation would make sense.


but if users are reading only one chapter per book,
then recommending new books becomes more important.

In [8]:
chapters_per_user_book = ( interactions .groupby(['user_id', 'book_id'])['chapter_id'] .nunique() )

print('\nchapters per user-book pair')
print(chapters_per_user_book.describe())


chapters per user-book pair
count    999520.000000
mean          1.000480
std           0.021909
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max           2.000000
Name: chapter_id, dtype: float64


checking percentage of user-book pairs
where only one chapter was read

In [9]:
single_chapter_percent = ( (chapters_per_user_book == 1).mean() * 100 )

print('\npercentage of user-book pairs with only one chapter interaction')
print(round(single_chapter_percent, 2), '%')


percentage of user-book pairs with only one chapter interaction
99.95 %


based on this analysis,
I found that users mostly interact with one chapter per book.


because of this,
I decided to build a content based recommendation system
for recommending new books.

MERGING DATASETS

interactions dataset only tells us:
- which user interacted
- which chapter was read


but the actual book information like genres/tags
exists inside chapters dataset.


so we need to merge both datasets together.

In [10]:
merged_data = interactions.merge( chapters[['chapter_id', 'book_id', 'tags']], on=['chapter_id', 'book_id'], how='left' )

print('\nmerged data shape:', merged_data.shape)
print('null tags after merge:', merged_data['tags'].isnull().sum())


merged data shape: (1000000, 4)
null tags after merge: 0


BUILDING USER TASTE PROFILE

now I want to understand user taste.


simple logic:
if a user reads many crime/thriller books,
then we can assume they like those genres.


important thing:
I am using unique user-book pairs.


why?
because I do not want repeated chapter interactions
from the same book to artificially increase genre weight.
keeping only unique user-book combinations

In [11]:
user_books = merged_data[['user_id', 'book_id', 'tags']].drop_duplicates()

In [12]:
# splitting tags

user_books['tags'] = user_books['tags'].fillna('').str.split('|')

In [13]:
# creating separate rows for each genre

user_genres = user_books.explode('tags')

In [14]:
# removing blank genres

user_genres = user_genres[ user_genres['tags'] != '' ]

creating user genre matrix
rows -> users
columns -> genres
values -> whether user interacted with that genre

In [15]:
user_profile = ( user_genres .groupby(['user_id', 'tags']) .size() .unstack(fill_value=0) )

print('\nuser profile shape:', user_profile.shape)
print(user_profile.head())


user profile shape: (149803, 15)
tags          Adventure  Crime  Dystopian  Fantasy  Graphic Novel  \
user_id                                                             
user_0000013          2      0          0        0              2   
user_0000115          0      0          1        1              2   
user_0000177          1      0          1        0              0   
user_0000188          2      0          1        1              0   
user_0000257          3      1          3        1              1   

tags          Historical Fiction  Horror  Humor  Literary Fiction  Mystery  \
user_id                                                                      
user_0000013                   2       1      2                 1        1   
user_0000115                   3       1      1                 1        0   
user_0000177                   1       3      2                 1        0   
user_0000188                   0       1      2                 0        1   
user_0000257  

BUILDING BOOK PROFILE

now creating genre profile for books.


 counting how many times each genre appears
 across all chapters of a book.

 this keeps it consistent with user profile
 which also uses genre counts.

In [16]:
book_data = chapters[['book_id', 'tags']].drop_duplicates()

In [17]:
# splitting tags

book_data['tags'] = book_data['tags'].fillna('').str.split('|')

In [18]:
# separate row for every genre

book_genres = book_data.explode('tags')

In [19]:
# remove blanks

book_genres = book_genres[ book_genres['tags'] != '' ]

In [20]:
# creating book genre matrix

book_profile = (
    book_genres
    .groupby(['book_id', 'tags'])
    .size()
    .unstack(fill_value=0)
)

print('\nbook profile shape:', book_profile.shape)
print(book_profile.head())


book profile shape: (9575, 15)
tags     Adventure  Crime  Dystopian  Fantasy  Graphic Novel  \
book_id                                                        
100089           0      0          0        2              0   
100096           0      0          0        0              0   
100193           0      1          1        1              0   
100205           4      0          0        1              0   
100301           0      0          2        0              0   

tags     Historical Fiction  Horror  Humor  Literary Fiction  Mystery  \
book_id                                                                 
100089                    0       0      0                 0        0   
100096                    0       0      0                 0        0   
100193                    0       0      1                 1        1   
100205                    0       1      0                 1        1   
100301                    0       0      3                 0        0   

tags   

ALIGNING USER AND BOOK MATRICES

cosine similarity only works properly
when both matrices have same genre columns.


so here I am making sure that:
- user profile columns
- book profile columns
are exactly same.

In [21]:
all_genres = sorted( list( set(user_profile.columns) .union(set(book_profile.columns)) ) )

for genre in all_genres:
  if genre not in user_profile.columns:
    user_profile[genre] = 0

if genre not in book_profile.columns:
    book_profile[genre] = 0

In [22]:
# keeping same column order

user_profile = user_profile[all_genres]
book_profile = book_profile[all_genres]

print('\nuser profile final shape:', user_profile.shape)
print('book profile final shape:', book_profile.shape)


user profile final shape: (149803, 15)
book profile final shape: (9575, 15)


POPULAR BOOKS FOR COLD START USERS

cold start means completely new users.


for new users we do not have reading history.
so we cannot understand their taste.


in that case,
recommending popular books is a practical fallback solution.

In [23]:
popular_books = ( interactions .groupby('book_id') .size() .sort_values(ascending=False) )

popular_books = popular_books.index.tolist()

print('\nmost popular books')
print(popular_books[:10])


most popular books
[672233, 615276, 760434, 756944, 395485, 791274, 181288, 682667, 316800, 478798]


RECOMMENDATION FUNCTION

this is the main recommendation function.


steps:
1. get user taste profile
2. compare it with all books
3. find most similar books
4. remove already read books
5. return top recommendations

In [24]:
def recommend_books(user_id, top_n=10):
  # checking if user exists
  # if user is new,
  # return popular books

  if user_id not in user_profile.index:

    print('new user found -> returning popular books')

    return popular_books[:top_n]


  # getting user vector
  user_vector = user_profile.loc[user_id].values.reshape(1, -1)


  # comparing user taste with all books
  sim_scores = cosine_similarity(
    user_vector,
    book_profile.values
  )[0]


  # converting similarity scores into series
  sim_scores = pd.Series(
    sim_scores,
    index=book_profile.index
  )


  # no point recommending books user already knows
  already_read = set(
    interactions[
      interactions['user_id'] == user_id
    ]['book_id'].unique()
  )


  # removing already read books
  sim_scores = sim_scores[
    ~sim_scores.index.isin(already_read)
  ]


  # getting top similar books
  recommended = (
    sim_scores
    .sort_values(ascending=False)
    .head(top_n)
    .index
    .tolist()
  )


  return recommended

TESTING THE MODEL

testing recommendation system on one sample user
just to check whether recommendations are coming properly.

In [25]:
sample_user = interactions['user_id'].iloc[0]

print('\nsample user:', sample_user)


sample user: user_2378720


In [26]:
# checking books already read by user

already_read_books = interactions[ interactions['user_id'] == sample_user ]['book_id'].unique()

print('\nbooks already interacted by user')
print(already_read_books)


books already interacted by user
[444295 724473 562543 309174 176115]


In [27]:
#generating recommendations

recommendations = recommend_books(sample_user, top_n=10)

print('\nrecommended books')
print(recommendations)


recommended books
[911259, 983687, 603518, 927251, 829630, 510103, 137042, 660625, 896456, 319703]


In [28]:
# checking cold start user

print('\nchecking cold start user')
print(recommend_books('new_user_123'))


checking cold start user
new user found -> returning popular books
[672233, 615276, 760434, 756944, 395485, 791274, 181288, 682667, 316800, 478798]


MODEL EVALUATION


 now checking whether recommendations are actually useful.

 evaluation method:
 - remove one book from user history
 - generate recommendations using remaining books
 - check whether hidden book comes back in top recommendations

 this is called Hit Rate @ K.

 simple meaning:
 if hidden book comes inside top 10 recommendations,
 we count it as a successful recommendation.

In [29]:
def evaluate_model(sample_size=2000, k=10):


    # taking only users with more than 1 book
    # because:
    # one book will be hidden for testing
    # remaining books will be used for recommendation

    valid_users = (
        interactions
        .groupby('user_id')['book_id']
        .nunique()
    )


    valid_users = valid_users[
        valid_users > 1
    ].index.tolist()


    # random sample for faster testing

    np.random.seed(42)


    sample_users = np.random.choice(
        valid_users,
        size=min(sample_size, len(valid_users)),
        replace=False
    )


    hits = 0
    total = 0


    print('\nstarting model evaluation...')


    for user_id in sample_users:


        # getting books interacted by user

        user_books = interactions[
            interactions['user_id'] == user_id
        ]['book_id'].unique()


        # skipping users with very small history

        if len(user_books) < 2:
            continue


        # randomly hiding one book

        hidden_book = np.random.choice(user_books)


        # remaining books after removing hidden book

        remaining_books = [
            book for book in user_books
            if book != hidden_book
        ]


        # creating temporary data using remaining books

        temp_data = merged_data[
            (merged_data['user_id'] == user_id) &
            (merged_data['book_id'].isin(remaining_books))
        ]


        # keeping only required columns

        temp_data = temp_data[['book_id', 'tags']].drop_duplicates()


        # splitting genres

        temp_data['tags'] = (
            temp_data['tags']
            .fillna('')
            .str.split('|')
        )


        # converting one row with many genres
        # into multiple rows with single genre

        temp_data = temp_data.explode('tags')


        # removing blank values

        temp_data = temp_data[
            temp_data['tags'] != ''
        ]


        # if no genres found then skip user

        if len(temp_data) == 0:
            continue


        # creating temporary user taste vector

        temp_vector = pd.Series(0, index=all_genres)


        # marking genres user has interacted with

        for genre in temp_data['tags']:

            if genre in temp_vector.index:
                temp_vector[genre] += 1


        # checking similarity with all books

        sims = cosine_similarity(
            temp_vector.values.reshape(1, -1),
            book_profile.values
        )[0]


        sims = pd.Series(
            sims,
            index=book_profile.index
        )


        # removing books already known to user

        sims = sims[
            ~sims.index.isin(remaining_books)
        ]


        # taking top recommendations

        top_books = (
            sims
            .sort_values(ascending=False)
            .head(k)
            .index
            .tolist()
        )


        # checking whether hidden book came back

        if hidden_book in top_books:
            hits += 1


        total += 1


    # final hit rate

    hit_rate = hits / total if total > 0 else 0


    print('\nmodel evaluation results')
    print('users evaluated:', total)
    print('successful hits:', hits)
    print(f'hit rate @{k}:', round(hit_rate, 4))
    print(f'percentage:', round(hit_rate * 100, 2), '%')


    return hit_rate


# running evaluation

hr = evaluate_model(sample_size=2000, k=10)



starting model evaluation...

model evaluation results
users evaluated: 2000
successful hits: 4
hit rate @10: 0.002
percentage: 0.2 %


 GENERATING FINAL RECOMMENDATIONS


 now generating recommendations for all users.

 final recommendations will be saved into csv file
 so that they can be checked later easily.

In [31]:


print('\ngenerating recommendations for users...')


final_output = []


all_users = interactions['user_id'].unique()[:500]  # generating recommendations for sample users only
                                                    # because generating for all 150k users takes longer time
                                                    # and the main goal here is to demonstrate recommendation logic


for user_id in all_users:


    # getting recommendations

    recs = recommend_books(user_id, top_n=10)


    final_output.append({
        'user_id': user_id,
        'recommended_books': '|'.join(map(str, recs))
    })


# converting into dataframe

final_df = pd.DataFrame(final_output)


# saving recommendation file

final_df.to_csv(
    'user_recommendations.csv',
    index=False
)


print('\nrecommendation file saved successfully')
print(final_df.head())


generating recommendations for all users...

recommendation file saved successfully
        user_id                                  recommended_books
0  user_2378720  911259|983687|603518|927251|829630|510103|1370...
1  user_2321122  415201|191876|697978|940946|207348|721678|3158...
2  user_2335775  952353|266309|107568|206105|848851|656921|4881...
3  user_7906001  374721|769102|781180|848244|535320|704349|5967...
4  user_9981689  752801|105409|265001|544692|834815|487380|1885...


Tradeoffs

One important decision in this project was choosing content-based filtering instead of collaborative filtering.

Collaborative filtering usually works by finding users with similar reading behavior and recommending books liked by similar users. But after exploring the dataset, I found that most users had very limited interaction history. On average, users interacted with only around 6 to 7 books. Because of this, it becomes difficult to find strong similarities between users.

That is why I chose content-based filtering. This approach works better even when user history is small because it focuses more on the genres and content the user has already interacted with instead of comparing users with each other.

At the same time, this approach also has some limitations. Since recommendations are generated mainly from the user’s existing genre preferences, the system mostly recommends books similar to what the user already reads. It may not recommend completely new or unexpected genres very often.

Another tradeoff was how interactions were treated. In this dataset, every interaction was treated equally because we do not have extra information like ratings, reading time, or completion percentage. In reality, a user who reads many chapters from the same book may be more interested than someone who reads only one chapter, but this difference cannot be captured properly with the current data.

So overall, the model was designed to stay simple, practical, and aligned with the information available in the dataset.
I kept the following things simple intentionally

Future Improvements

There are many things that could improve this recommendation system further in the future.

One improvement would be using TF-IDF weighting instead of simple genre counts. Right now, very common genres like Crime or Romance appear frequently across many books, so they may dominate recommendations. TF-IDF could help give more importance to less common genres and create more personalized recommendations.

Another improvement would be trying a hybrid recommendation system by combining content-based filtering with collaborative filtering. At the moment, collaborative filtering is difficult because user interaction history is very limited. But if the platform collects more interactions over time, collaborative filtering could become much more useful.

The biggest improvement would come from better user data. Currently, the dataset only contains interaction history, which is an example of implicit feedback. We do not know whether the user actually liked the chapter, finished reading it, or spent a lot of time on it. If we had information like ratings, timestamps, reading completion percentage, or reading time, the recommendation quality could improve significantly.

I would also like to experiment with author-based recommendations in the future. If a user repeatedly interacts with books from the same author, that can become another strong signal for recommendation apart from genres alone